# Conv2d : appliquer le même petit filtre à plusieurs positions

Calculer les produits et leur somme avant de suivre les canaux d’un réseau.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#convolution-a-la-main).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Une couche Linear combine toutes les coordonnées d’une ligne. Une convolution réutilise un petit filtre local, appelé noyau, en le déplaçant sur une grille.

Nous choisissons une grille pédagogique de trois lignes et trois colonnes. Les lignes représentent des positions fréquentielles, les colonnes des instants ; ces entiers ne sont pas des mesures en dB.

| Bande / temps | t0 | t1 | t2 |
| --- | --- | --- | --- |
| f0 | 1 | 2 | 3 |
| f1 | 4 | 5 | 6 |
| f2 | 7 | 8 | 9 |

La première fenêtre contient [[1,2],[4,5]]. Le noyau choisi vaut [[1,0],[0,−1]].

À la première position, les produits sont 1×1, 2×0, 4×0 et 5×(−1). Leur somme vaut −4. Nous déplaçons ensuite le même noyau d’une colonne, puis d’une ligne.

La carte porte quatre axes : un extrait, un canal, trois bandes, trois instants. Le noyau porte un canal de sortie, un canal d’entrée et une grille 2×2.

## Démonstration 1 · Du produit local à la carte de sortie

Les poids du filtre sont choisis à la main pour rendre le calcul visible.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Les crochets extérieurs créent les axes extrait et canal, tous deux de taille un. Les deux derniers axes contiennent les valeurs de la grille.

In [4]:
carte = torch.tensor([[[[1., 2., 3.],
                        [4., 5., 6.],
                        [7., 8., 9.]]]])
noyau = torch.tensor([[[[1., 0.], [0., -1.]]]])


In [5]:
observer('carte', carte)
observer('noyau', noyau)

carte forme (1, 1, 3, 3) type torch.float32 requires_grad False
  valeurs : [[[[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]]]
noyau forme (1, 1, 2, 2) type torch.float32 requires_grad False
  valeurs : [[[[1.0, 0.0], [0.0, -1.0]]]]


### Calculer l’état suivant

[0,0] choisit l’extrait et le canal ; :2,:2 choisit les deux premières lignes et colonnes. La multiplication agit case par case.

In [6]:
fenetre = carte[0, 0, :2, :2]
produits = fenetre * noyau[0, 0]


In [7]:
observer('fenetre', fenetre)
observer('produits', produits)

fenetre forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 2.0], [4.0, 5.0]]
produits forme (2, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 0.0], [0.0, -5.0]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [8]:
print("produits premiere case :", produits.tolist())
print("somme :", produits.sum().item())


produits premiere case : [[1.0, 0.0], [0.0, -5.0]]
somme : -4.0


### Calculer l’état suivant

Conv2d(1,1,...) reçoit un canal et produit un canal. kernel_size=2 fixe le filtre 2×2, stride=1 son pas, padding=0 l’absence de bord ajouté.

In [9]:
couche = nn.Conv2d(1, 1, kernel_size=2, stride=1, padding=0, bias=False)


In [10]:
observer('couche', couche)

### Appliquer le contexte explicite

copy_ écrit le noyau dans les poids de la couche. Cette affectation manuelle dans no_grad n’est pas un entraînement.

In [11]:
with torch.no_grad():
    couche.weight.copy_(noyau)


### Calculer l’état suivant

Sans padding, un filtre de deux cases peut commencer à deux positions sur un axe de taille trois. La sortie spatiale est donc 2×2.

In [12]:
sortie = couche(carte)


In [13]:
observer('sortie', sortie)

sortie forme (1, 1, 2, 2) type torch.float32 requires_grad True
  valeurs : [[[[-4.0, -4.0], [-4.0, -4.0]]]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [14]:
print("forme :", tuple(sortie.shape))
print("sortie :", sortie[0, 0].tolist())


forme : (1, 1, 2, 2)
sortie : [[-4.0, -4.0], [-4.0, -4.0]]


### Appliquer le contexte explicite

Le premier poids devient 2. À la première position, 2×1−5 vaut maintenant −3. Le même changement agit sur toutes les fenêtres.

In [15]:
with torch.no_grad():
    couche.weight[0, 0, 0, 0] = 2.


### Calculer l’état suivant

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [16]:
variante = couche(carte)


In [17]:
observer('variante', variante)

variante forme (1, 1, 2, 2) type torch.float32 requires_grad True
  valeurs : [[[[-3.0, -2.0], [0.0, 1.0]]]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [18]:
print("apres modification du premier poids :", variante[0, 0].tolist())

apres modification du premier poids : [[-3.0, -2.0], [0.0, 1.0]]


Les poids sont partagés entre positions, mais chaque sortie dépend des valeurs de sa propre fenêtre.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Calculez le premier résultat d’un noyau [[1,0],[0,−1]] appliqué à la grille 1 à 9.

In [19]:
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.tensor([[[[1.,0.],[0.,-1.]]]])
exo_valeur = ...  # Remplacer les points par votre réponse.
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_valeur)-torch.nn.functional.conv2d(exo_x,exo_noyau)[0,0,0,0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

La première fenêtre est [[1,2],[4,5]]. Faites les quatre produits, puis leur somme.

</details>

### Exercice 2

Avec le même noyau 2×2 et stride=2, quelle forme de sortie reste-t-il sur une entrée 3×3 ?

In [20]:
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.ones(1,1,2,2)
exo_forme = ...  # Remplacer les points par votre réponse.
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.functional.conv2d(exo_x,exo_noyau,stride=2).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Le premier placement couvre indices 0 et 1. Un nouveau départ à 2 dépasserait le bord sans padding.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.tensor([[[[1.,0.],[0.,-1.]]]])
exo_valeur = -4.
if exo_valeur is Ellipsis or exo_valeur is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert abs(float(exo_valeur)-torch.nn.functional.conv2d(exo_x,exo_noyau)[0,0,0,0].item()) < 1e-6
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_x = torch.arange(1.,10.).reshape(1,1,3,3)
exo_noyau = torch.ones(1,1,2,2)
exo_forme = (1,1,1,1)
if exo_forme is Ellipsis or exo_forme is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert tuple(exo_forme) == tuple(torch.nn.functional.conv2d(exo_x,exo_noyau,stride=2).shape)
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [21]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()